# 🐭 Mouse Video Processor
**Pipeline:** Load folder → Draw crop regions → Set start/stop times → Trim & crop → Concatenate side-by-side

---

## Part 1: Setup & Configuration

In [21]:
%matplotlib widget

In [4]:
# ── Install dependencies if needed ──────────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['ipywidgets', 'ipympl', 'matplotlib', 'opencv-python-headless', 'numpy', 'Pillow']:
    try:
        __import__(pkg.split('-')[0].replace('opencv', 'cv2').replace('Pillow', 'PIL'))
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)

print('✅ All dependencies ready.')

✅ All dependencies ready.


In [5]:
import os, json, subprocess, shutil, time, re
from datetime import datetime
from pathlib import Path

import numpy as np
import cv2
import matplotlib
matplotlib.use('widget')  # enables interactive matplotlib in Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.widgets import PolygonSelector
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print('✅ Imports successful.')

✅ Imports successful.


In [6]:
# ── GPU Detection ────────────────────────────────────────────────────────────
def detect_gpu():
    """Check for NVIDIA GPU / nvenc support via ffmpeg."""
    try:
        result = subprocess.run(
            ['ffmpeg', '-hide_banner', '-encoders'],
            capture_output=True, text=True, timeout=10
        )
        if 'h264_nvenc' in result.stdout:
            # Double-check by actually trying a quick encode
            test = subprocess.run(
                ['ffmpeg', '-f', 'lavfi', '-i', 'nullsrc=s=64x64:d=1',
                 '-c:v', 'h264_nvenc', '-f', 'null', '-'],
                capture_output=True, timeout=15
            )
            return test.returncode == 0
    except Exception:
        pass
    return False

def check_ffmpeg():
    try:
        subprocess.run(['ffmpeg', '-version'], capture_output=True, timeout=5)
        return True
    except FileNotFoundError:
        return False

FFMPEG_AVAILABLE = check_ffmpeg()
GPU_AVAILABLE = detect_gpu() if FFMPEG_AVAILABLE else False

if not FFMPEG_AVAILABLE:
    print('❌ ffmpeg not found. Please install ffmpeg and ensure it is on your PATH.')
elif GPU_AVAILABLE:
    print('✅ GPU detected — using h264_nvenc for fast encoding.')
    VIDEO_CODEC = 'h264_nvenc'
    CODEC_OPTS  = ['-rc', 'constqp', '-cq', '20']
else:
    print('⚠️  WARNING: No NVIDIA GPU / nvenc support detected — falling back to CPU (libx264).')
    print('   Processing will be significantly slower.')
    VIDEO_CODEC = 'libx264'
    CODEC_OPTS  = ['-crf', '20', '-preset', 'fast']

# ── ffmpeg version ───────────────────────────────────────────────────────────
if FFMPEG_AVAILABLE:
    ver = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    print(f'   ffmpeg: {ver.stdout.splitlines()[0]}')

⚠️  WARNING: No NVIDIA GPU / nvenc support detected — falling back to CPU (libx264).
   Processing will be significantly slower.
   ffmpeg: ffmpeg version 8.1.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers


In [7]:
# ── Folder Selection ─────────────────────────────────────────────────────────
VIDEO_EXTENSIONS = {'.avi', '.mp4', '.mov', '.mkv', '.wmv'}

folder_widget = widgets.Text(
    value='',
    placeholder='e.g. C:/Users/you/Videos/Session01  or  /data/mice/session01',
    description='Video folder:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '120px'}
)
output_widget = widgets.Text(
    value='',
    placeholder='Leave blank to auto-create "processed" subfolder',
    description='Output folder:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': '120px'}
)
scan_btn = widgets.Button(description='📂 Scan Folder', button_style='primary')
scan_out = widgets.Output()

# Global state
STATE = {
    'input_folder': None,
    'output_folder': None,
    'video_files': [],
    'session_config': [],   # list of per-video configs
    'current_video_idx': 0,
}

def scan_folder(btn):
    with scan_out:
        clear_output()
        folder = Path(folder_widget.value.strip())
        if not folder.exists():
            print(f'❌ Folder not found: {folder}')
            return
        videos = sorted([f for f in folder.iterdir()
                         if f.suffix.lower() in VIDEO_EXTENSIONS])
        if not videos:
            print(f'⚠️  No video files found in {folder}')
            return

        STATE['input_folder'] = folder
        out_folder = output_widget.value.strip()
        STATE['output_folder'] = Path(out_folder) if out_folder else folder / 'processed'
        STATE['output_folder'].mkdir(parents=True, exist_ok=True)
        STATE['video_files'] = videos
        STATE['session_config'] = []
        STATE['current_video_idx'] = 0

        print(f'✅ Found {len(videos)} video(s) in: {folder}')
        print(f'   Output → {STATE["output_folder"]}')
        print()
        for i, v in enumerate(videos):
            print(f'  [{i+1}] {v.name}')
        print()
        print('➡️  Proceed to Part 2 to draw crop regions.')

scan_btn.on_click(scan_folder)
display(folder_widget, output_widget, scan_btn, scan_out)

Text(value='', description='Video folder:', layout=Layout(width='600px'), placeholder='e.g. C:/Users/you/Video…

Text(value='', description='Output folder:', layout=Layout(width='600px'), placeholder='Leave blank to auto-cr…

Button(button_style='primary', description='📂 Scan Folder', style=ButtonStyle())

Output()

---
## Part 2: Interactive Crop Region Editor

For each video you will:
1. See a frame grabbed ~10 min in
2. Click **Add Crop Region** and draw a polygon by clicking corners (close it by clicking the first point again or pressing Enter)
3. Enter the MousePair ID and camera type (side/bottom) for that crop
4. Repeat for all arenas in this video
5. Click **Confirm & Next Video** when done

In [22]:
# ── Frame extraction utility ─────────────────────────────────────────────────
def get_video_duration(video_path):
    """Return duration in seconds using ffprobe."""
    try:
        result = subprocess.run([
            'ffprobe', '-v', 'quiet', '-print_format', 'json',
            '-show_streams', str(video_path)
        ], capture_output=True, text=True, timeout=15)
        info = json.loads(result.stdout)
        for stream in info.get('streams', []):
            if stream.get('codec_type') == 'video':
                dur = stream.get('duration')
                if dur:
                    return float(dur)
        # fallback: format duration
        result2 = subprocess.run([
            'ffprobe', '-v', 'quiet', '-print_format', 'json',
            '-show_format', str(video_path)
        ], capture_output=True, text=True, timeout=15)
        fmt = json.loads(result2.stdout)
        return float(fmt.get('format', {}).get('duration', 0))
    except Exception:
        return 0

def extract_frame(video_path, target_seconds=600, tmp_dir=None):
    """Extract a single frame at target_seconds (fallback to 30s, then 0s)."""
    video_path = Path(video_path)
    if tmp_dir is None:
        tmp_dir = video_path.parent
    out_frame = Path(tmp_dir) / f'_frame_{video_path.stem}.jpg'

    duration = get_video_duration(video_path)
    candidates = [target_seconds, 30, 0]
    seek_time = next((t for t in candidates if t < duration or t == 0), 0)

    label = f'{seek_time//60:.0f}m{seek_time%60:.0f}s'
    print(f'  Grabbing frame at {label} (video length: {duration/60:.1f} min)...')

    cmd = [
        'ffmpeg', '-y', '-ss', str(seek_time),
        '-i', str(video_path),
        '-frames:v', '1', '-q:v', '2',
        str(out_frame)
    ]
    subprocess.run(cmd, capture_output=True, timeout=30)
    if not out_frame.exists():
        raise FileNotFoundError(f'Frame extraction failed for {video_path.name}')
    img = cv2.imread(str(out_frame))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img, seek_time

print('✅ Frame extraction utilities ready.')

✅ Frame extraction utilities ready.


In [23]:
# ── Crop Editor Class ────────────────────────────────────────────────────────
class CropEditor:
    MAX_CROPS = 8

    COLORS = [
        '#FF4444', '#44AAFF', '#44FF88', '#FFAA00',
        '#FF44FF', '#00DDDD', '#FFFF44', '#FF8844'
    ]

    def __init__(self, video_path, output):
        self.video_path = Path(video_path)
        self.output = output  # ipywidgets Output for messages
        self.crops = []       # list of dicts: {points, pair_id, camera, color, patch}
        self._selector = None
        self._fig = None
        self._ax = None
        self._drawing = False
        self._frame = None
        self._frame_time = None

    def launch(self):
        with self.output:
            clear_output()
            print(f'Loading frame from: {self.video_path.name}')

            try:
                self._frame, self._frame_time = extract_frame(
                    self.video_path,
                    target_seconds=600,
                    tmp_dir=STATE['output_folder']
                )
            except Exception as e:
                with self.output:
                    print(f'❌ {e}')
                return

            h, w = self._frame.shape[:2]
            figw = min(16, w / 80)
            figh = figw * h / w
    
            self._fig, self._ax = plt.subplots(figsize=(figw, figh))
            self._ax.imshow(self._frame)
            self._ax.set_title(
                f'{self.video_path.name}  |  Frame at '
                f'{int(self._frame_time)//60}m{int(self._frame_time)%60:02d}s  |  '
                f'{len(self.crops)}/{self.MAX_CROPS} crop(s) defined',
                fontsize=9
            )
            self._ax.axis('off')
            self._fig.tight_layout()
            plt.show()
    
            self._build_controls()

    def _build_controls(self):
        self._add_btn = widgets.Button(
            description='➕ Add Crop Region',
            button_style='success',
            layout=widgets.Layout(width='180px')
        )
        self._undo_btn = widgets.Button(
            description='↩ Remove Last',
            button_style='warning',
            layout=widgets.Layout(width='140px')
        )
        self._confirm_btn = widgets.Button(
            description='✅ Confirm & Next Video',
            button_style='primary',
            layout=widgets.Layout(width='200px')
        )
        self._status = widgets.HTML(value='<i>Click "Add Crop Region" then draw a polygon on the image.</i>')

        # Per-crop entry form (hidden until polygon drawn)
        self._pair_id_box = widgets.Text(
            placeholder='e.g. M1_2 or PairA',
            description='MousePair ID:',
            layout=widgets.Layout(width='280px'),
            style={'description_width': '100px'}
        )
        self._camera_box = widgets.ToggleButtons(
            options=['side', 'bottom'],
            description='Camera:',
            button_style=''
        )
        self._save_crop_btn = widgets.Button(
            description='💾 Save This Crop',
            button_style='info',
            layout=widgets.Layout(width='160px')
        )
        self._crop_form = widgets.VBox([
            widgets.HTML('<b>Label this crop region:</b>'),
            self._pair_id_box,
            self._camera_box,
            self._save_crop_btn
        ], layout=widgets.Layout(
            border='1px solid #ccc', padding='10px',
            margin='5px 0', display='none'
        ))

        self._add_btn.on_click(self._on_add)
        self._undo_btn.on_click(self._on_undo)
        self._confirm_btn.on_click(self._on_confirm)
        self._save_crop_btn.on_click(self._on_save_crop)

        self._pending_points = None  # polygon points waiting to be labeled

        display(widgets.HBox([self._add_btn, self._undo_btn, self._confirm_btn]))
        display(self._status)
        display(self._crop_form)
        #display(self.output)

    def _on_add(self, btn):
        if len(self.crops) >= self.MAX_CROPS:
            self._status.value = f'⚠️  Maximum of {self.MAX_CROPS} crops reached.'
            return
        if self._drawing:
            self._status.value = '⚠️  Finish the current polygon first.'
            return

        color = self.COLORS[len(self.crops)]
        self._status.value = (
            f'<span style="color:{color}">'
            f'🖊 Drawing crop #{len(self.crops)+1}: click to add corners. '
            f'Press <b>Enter</b> or click first point to close.</span>'
        )
        self._drawing = True
        self._current_color = color

        self._selector = PolygonSelector(
            self._ax,
            onselect=self._on_polygon_done,
            props=dict(color=color, linewidth=2),
            handle_props=dict(markersize=6, markerfacecolor=color)
        )
        self._fig.canvas.draw_idle()

    def _on_polygon_done(self, verts):
        """Called when polygon is closed by user."""
        self._drawing = False
        self._pending_points = verts

        # Preview the polygon
        pts = np.array(verts)
        poly = plt.Polygon(pts, fill=False,
                           edgecolor=self._current_color,
                           linewidth=2, linestyle='--', alpha=0.8)
        self._ax.add_patch(poly)
        n = len(self.crops) + 1
        cx, cy = pts.mean(axis=0)
        self._ax.text(cx, cy, f'#{n}', color=self._current_color,
                      fontsize=12, fontweight='bold', ha='center', va='center')
        self._fig.canvas.draw_idle()

        # Remove PolygonSelector
        if self._selector:
            self._selector.disconnect_events()
            self._selector = None

        # Show label form
        self._pair_id_box.value = ''
        self._crop_form.layout.display = ''
        self._status.value = '<b>✏️  Enter the MousePair ID and camera type below, then click Save.</b>'

    def _on_save_crop(self, btn):
        pair_id = self._pair_id_box.value.strip()
        camera  = self._camera_box.value

        if not pair_id:
            self._status.value = '⚠️  Please enter a MousePair ID before saving.'
            return

        # Check for duplicate pair_id + camera combo
        existing = [(c['pair_id'], c['camera']) for c in self.crops]
        if (pair_id, camera) in existing:
            self._status.value = f'⚠️  {pair_id}_{camera} already defined. Use a different ID or camera.'
            return

        self.crops.append({
            'pair_id': pair_id,
            'camera': camera,
            'points': self._pending_points,
            'color': self._current_color,
        })
        self._pending_points = None
        self._crop_form.layout.display = 'none'

        # Update title
        self._ax.set_title(
            f'{self.video_path.name}  |  '
            f'{len(self.crops)}/{self.MAX_CROPS} crop(s) defined',
            fontsize=9
        )
        self._fig.canvas.draw_idle()

        crop_list = ', '.join(f"{c['pair_id']}_{c['camera']}" for c in self.crops)
        self._status.value = (
            f'✅ Saved: <b>{pair_id}_{camera}</b>. '
            f'Crops so far: {crop_list}. '
            f'Add more or click Confirm & Next Video.'
        )

    def _on_undo(self, btn):
        if not self.crops:
            self._status.value = 'Nothing to remove.'
            return
        removed = self.crops.pop()
        # Redraw frame without last crop
        self._ax.cla()
        self._ax.imshow(self._frame)
        self._ax.axis('off')
        for i, c in enumerate(self.crops):
            pts = np.array(c['points'])
            poly = plt.Polygon(pts, fill=False,
                               edgecolor=c['color'], linewidth=2, linestyle='--')
            self._ax.add_patch(poly)
            cx, cy = pts.mean(axis=0)
            self._ax.text(cx, cy, f'#{i+1}', color=c['color'],
                          fontsize=12, fontweight='bold', ha='center', va='center')
        self._ax.set_title(
            f'{self.video_path.name}  |  {len(self.crops)}/{self.MAX_CROPS} crop(s) defined',
            fontsize=9
        )
        self._fig.canvas.draw_idle()
        self._status.value = f'↩ Removed crop: {removed["pair_id"]}_{removed["camera"]}'

    def _on_confirm(self, btn):
        if not self.crops:
            self._status.value = '⚠️  No crops defined. Add at least one region.'
            return
        plt.close(self._fig)
        self._status.value = f'✅ Confirmed {len(self.crops)} crop(s) for {self.video_path.name}.'
        self._confirm_callback(self.crops)

    def set_confirm_callback(self, fn):
        self._confirm_callback = fn

print('✅ CropEditor class ready.')

✅ CropEditor class ready.


In [28]:
# ── Launch Crop Editor for current video ─────────────────────────────────────
crop_output = widgets.Output()
display(crop_output)

def run_crop_editor_for_current():
    idx = STATE['current_video_idx']
    videos = STATE['video_files']

    if not videos:
        print('❌ No videos loaded. Run Part 1 first.')
        return
    if idx >= len(videos):
        print('✅ All videos have been cropped. Proceed to Part 3.')
        return

    video = videos[idx]
    print(f'\n📹 Video {idx+1} of {len(videos)}: {video.name}')

    editor = CropEditor(video, crop_output)

    def on_confirmed(crops):
        STATE['session_config'].append({
            'video_path': str(video),
            'video_name': video.name,
            'crops': crops
        })
        STATE['current_video_idx'] += 1
        next_idx = STATE['current_video_idx']
        if next_idx < len(videos):
            print(f'\n➡️  Re-run this cell to load the next video: {videos[next_idx].name}')
        else:
            print('\n✅ All videos cropped! Proceed to Part 3 to set start/stop times.')

    editor.set_confirm_callback(on_confirmed)
    editor.launch()

run_crop_editor_for_current()

Output()

✅ All videos have been cropped. Proceed to Part 3.


> **Re-run the cell above** to load each subsequent video. When all videos are done, proceed to Part 3.

---
## Part 3: Start/Stop Times → Trim & Crop

For each MousePair crop you defined, enter a **start time** and **duration**. Then run the ffmpeg trim + crop pipeline.

In [29]:
# ── Build the timing input form ──────────────────────────────────────────────
timing_widgets = {}   # keyed by (video_name, pair_id, camera)
timing_boxes   = []

if not STATE['session_config']:
    print('❌ No crop data found. Complete Part 2 first.')
else:
    print('Enter start time (MM:SS or HH:MM:SS) and duration (MM:SS or HH:MM:SS) for each crop.\n')

    for vid_cfg in STATE['session_config']:
        vname = vid_cfg['video_name']
        header = widgets.HTML(f'<h4 style="margin-bottom:4px">📹 {vname}</h4>')
        rows = [header]

        for crop in vid_cfg['crops']:
            pid    = crop['pair_id']
            cam    = crop['camera']
            color  = crop.get('color', '#888')
            label  = f'<span style="color:{color}">■</span> <b>{pid}_{cam}</b>'

            start_w = widgets.Text(
                value='00:00',
                description='Start:',
                layout=widgets.Layout(width='200px'),
                style={'description_width': '50px'}
            )
            dur_w = widgets.Text(
                value='45:00',
                description='Duration:',
                layout=widgets.Layout(width='220px'),
                style={'description_width': '70px'}
            )
            key = (vname, pid, cam)
            timing_widgets[key] = {'start': start_w, 'duration': dur_w}

            row = widgets.HBox([
                widgets.HTML(f'<div style="width:200px;padding-top:6px">{label}</div>'),
                start_w, dur_w
            ])
            rows.append(row)

        box = widgets.VBox(rows, layout=widgets.Layout(
            border='1px solid #ddd', padding='10px', margin='5px 0'
        ))
        timing_boxes.append(box)

    display(widgets.VBox(timing_boxes))
    print('\nWhen done, run the next cell to estimate time and start processing.')

Enter start time (MM:SS or HH:MM:SS) and duration (MM:SS or HH:MM:SS) for each crop.




When done, run the next cell to estimate time and start processing.


In [45]:
# ── Time parsing & estimation utilities ─────────────────────────────────────
def parse_time(t):
    """Parse MM:SS or HH:MM:SS to total seconds."""
    t = t.strip()
    parts = t.split(':')
    if len(parts) == 2:
        return int(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    else:
        raise ValueError(f'Cannot parse time: {t}')

def seconds_to_ffmpeg(s):
    s = int(s)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{sec:02d}'

def estimate_processing_time(duration_secs, n_crops, gpu=GPU_AVAILABLE):
    """Rough estimate: GPU ~0.3x realtime, CPU ~1.5x realtime per crop."""
    factor = 0.3 if gpu else 1.5
    total = duration_secs * n_crops * factor
    m, s = divmod(int(total), 60)
    return f'~{m}m {s}s'

import numpy as np

def points_to_crop_filter(points, frame_w, frame_h, is_quad=True):
    """
    Convert polygon points to a standard ffmpeg bounding box crop string.
    No perspective warping is applied; it just extracts the raw rectangular region.
    """
    pts = np.array(points, dtype=np.float32)
    
    # 1. Calculate the bounding box for the drawn polygon
    x_min = int(np.floor(pts[:, 0].min()))
    y_min = int(np.floor(pts[:, 1].min()))
    x_max = int(np.ceil(pts[:, 0].max()))
    y_max = int(np.ceil(pts[:, 1].max()))
    
    # 2. Clamp to ensure we don't accidentally crop outside the video boundaries
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(frame_w, x_max)
    y_max = min(frame_h, y_max)
    
    w = x_max - x_min
    h = y_max - y_min
    
    # 3. Ensure even dimensions for h264 encoding (prevents the silver spike glitch)
    w = w - (w % 2)
    h = h - (h % 2)

    # 4. Return the standard crop filter
    filter_str = f'crop={w}:{h}:{x_min}:{y_min}'
    
    return filter_str, w, h

print('✅ Time & crop filter utilities ready.')

✅ Time & crop filter utilities ready.


In [46]:
# ── Estimate processing time ─────────────────────────────────────────────────
if timing_widgets:
    total_crops = sum(len(v['crops']) for v in STATE['session_config'])
    # Use median duration from widgets as estimate
    try:
        durations = [parse_time(w['duration'].value)
                     for w in timing_widgets.values()]
        avg_dur = np.mean(durations)
        est = estimate_processing_time(avg_dur, total_crops)
        mode = 'GPU (h264_nvenc)' if GPU_AVAILABLE else 'CPU (libx264) ⚠️'
        print(f'📊 Processing estimate:')
        print(f'   Total crops : {total_crops}')
        print(f'   Avg duration: {avg_dur/60:.1f} min per crop')
        print(f'   Encoder     : {mode}')
        print(f'   Estimated   : {est} total')
        print(f'   (Estimate assumes ~0.3x realtime GPU / ~1.5x realtime CPU)')
    except Exception as e:
        print(f'⚠️  Could not estimate: {e}')
else:
    print('❌ No timing data. Complete the form above first.')

📊 Processing estimate:
   Total crops : 8
   Avg duration: 5.0 min per crop
   Encoder     : CPU (libx264) ⚠️
   Estimated   : ~60m 0s total
   (Estimate assumes ~0.3x realtime GPU / ~1.5x realtime CPU)


In [47]:
# ── Main trim + crop execution ───────────────────────────────────────────────
process_out = widgets.Output()
process_btn = widgets.Button(
    description='🚀 Start Processing',
    button_style='danger',
    layout=widgets.Layout(width='200px')
)

processed_files = []   # will be populated for Part 4

def get_frame_size(video_path):
    cap = cv2.VideoCapture(str(video_path))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return w, h

def run_processing(btn):
    process_btn.disabled = True
    processed_files.clear()

    with process_out:
        clear_output()
        overall_start = time.time()

        for vid_cfg in STATE['session_config']:
            vpath = Path(vid_cfg['video_path'])
            vname = vid_cfg['video_name']
            print(f'\n{'='*60}')
            print(f'📹 Processing: {vname}')

            try:
                fw, fh = get_frame_size(vpath)
            except Exception as e:
                print(f'  ❌ Cannot read frame size: {e}'); continue

            for crop in vid_cfg['crops']:
                pid   = crop['pair_id']
                cam   = crop['camera']
                key   = (vname, pid, cam)
                tw    = timing_widgets.get(key)

                if tw is None:
                    print(f'  ⚠️  No timing found for {pid}_{cam}, skipping.')
                    continue

                try:
                    start_s = parse_time(tw['start'].value)
                    dur_s   = parse_time(tw['duration'].value)
                except ValueError as e:
                    print(f'  ❌ Time parse error for {pid}_{cam}: {e}'); continue

                out_name = f'{pid}_{cam}_trimmed.mp4'
                out_path = STATE['output_folder'] / out_name

                # Build crop filter
                pts = crop['points']
                filt, cw, ch = points_to_crop_filter(pts, fw, fh)

                cmd = [
                    'ffmpeg', '-y',
                    '-ss', seconds_to_ffmpeg(start_s),
                    '-i', str(vpath),
                    '-t', seconds_to_ffmpeg(dur_s),
                    '-vf', filt,
                    '-c:v', VIDEO_CODEC,
                    *CODEC_OPTS,
                    '-pix_fmt', 'yuv420p',
                    '-an',   # drop audio for behavioural videos
                    str(out_path)
                ]

                print(f'\n  ▶ {pid}_{cam}')
                print(f'    Start: {tw["start"].value}  |  Duration: {tw["duration"].value}')
                print(f'    Crop filter: {filt}')
                print(f'    Output: {out_name}')

                t0 = time.time()
                result = subprocess.run(cmd, capture_output=True, text=True)
                elapsed = time.time() - t0

                if result.returncode != 0:
                    print(f'    ❌ ffmpeg error:')
                    print(result.stderr[-800:])
                else:
                    size_mb = out_path.stat().st_size / 1e6
                    print(f'    ✅ Done in {elapsed:.1f}s  |  {size_mb:.1f} MB')
                    processed_files.append({
                        'pair_id': pid,
                        'camera': cam,
                        'path': str(out_path)
                    })

        # Save session config to txt
        save_session_txt()

        total_elapsed = time.time() - overall_start
        m, s = divmod(int(total_elapsed), 60)
        print(f'\n{'='*60}')
        print(f'✅ All done! Total time: {m}m {s}s')
        print(f'   Output folder: {STATE["output_folder"]}')
        print('➡️  Proceed to Part 4 to concatenate side-by-side.')

    process_btn.disabled = False

def save_session_txt():
    """Save session configuration to a .txt file for reproducibility."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    txt_path = STATE['output_folder'] / f'session_config_{ts}.txt'
    lines = [f'Mouse Video Processor — Session Config', f'Generated: {datetime.now()}', '']
    for vid_cfg in STATE['session_config']:
        lines.append(f'VIDEO: {vid_cfg["video_name"]}')
        for crop in vid_cfg['crops']:
            pid = crop['pair_id']; cam = crop['camera']
            key = (vid_cfg['video_name'], pid, cam)
            tw  = timing_widgets.get(key)
            start = tw['start'].value if tw else 'N/A'
            dur   = tw['duration'].value if tw else 'N/A'
            lines.append(f'  CROP: {pid}_{cam}')
            lines.append(f'    start={start}  duration={dur}')
            lines.append(f'    points={crop["points"]}')
        lines.append('')
    txt_path.write_text('\n'.join(lines))
    print(f'\n💾 Session config saved: {txt_path.name}')

process_btn.on_click(run_processing)
display(process_btn, process_out)

Button(button_style='danger', description='🚀 Start Processing', layout=Layout(width='200px'), style=ButtonStyl…

Output()

---
## Part 4: Concatenate Side-by-Side

For each MousePair ID, the `_side` and `_bottom` trimmed videos are merged side-by-side into a single output file.

In [48]:
# ── Build merge pairs ────────────────────────────────────────────────────────
merge_out = widgets.Output()
merge_btn = widgets.Button(
    description='🎬 Merge All Pairs',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

def run_merge(btn):
    merge_btn.disabled = True
    with merge_out:
        clear_output()

        # Group processed files by pair_id
        from collections import defaultdict
        pairs = defaultdict(dict)
        for f in processed_files:
            pairs[f['pair_id']][f['camera']] = f['path']

        # Also scan output folder for any _trimmed.mp4 files not in processed_files
        for mp4 in STATE['output_folder'].glob('*_trimmed.mp4'):
            stem = mp4.stem  # e.g. M1_2_side_trimmed
            for cam in ['side', 'bottom']:
                suffix = f'_{cam}_trimmed'
                if stem.endswith(suffix):
                    pid = stem[:-len(suffix)]
                    if cam not in pairs[pid]:
                        pairs[pid][cam] = str(mp4)

        if not pairs:
            print('❌ No processed files found. Run Part 3 first.')
            merge_btn.disabled = False
            return

        print(f'Found {len(pairs)} MousePair group(s):\n')
        overall_start = time.time()

        for pid, cameras in sorted(pairs.items()):
            print(f'{'='*55}')
            print(f'MousePair: {pid}')

            side_path   = cameras.get('side')
            bottom_path = cameras.get('bottom')

            if not side_path or not bottom_path:
                missing = 'side' if not side_path else 'bottom'
                print(f'  ⚠️  Missing {missing} video — skipping merge.')
                if side_path:   print(f'  Available: side   → {Path(side_path).name}')
                if bottom_path: print(f'  Available: bottom → {Path(bottom_path).name}')
                continue

            out_name = f'{pid}_merged.mp4'
            out_path = STATE['output_folder'] / out_name

            print(f'  Left  (side)  : {Path(side_path).name}')
            print(f'  Right (bottom): {Path(bottom_path).name}')
            print(f'  Output        : {out_name}')

            cmd = [
                'ffmpeg', '-y',
                '-i', side_path,
                '-i', bottom_path,
                '-filter_complex',
                '[0:v]scale=-2:1080[v0];[1:v]scale=-2:1080[v1];'
                '[v0][v1]hstack=inputs=2:shortest=0[vout]',
                '-map', '[vout]',
                '-c:v', VIDEO_CODEC,
                *CODEC_OPTS,
                '-pix_fmt', 'yuv420p',
                str(out_path)
            ]

            t0 = time.time()
            result = subprocess.run(cmd, capture_output=True, text=True)
            elapsed = time.time() - t0

            if result.returncode != 0:
                print(f'  ❌ ffmpeg error:')
                print(result.stderr[-600:])
            else:
                size_mb = out_path.stat().st_size / 1e6
                print(f'  ✅ Merged in {elapsed:.1f}s  |  {size_mb:.1f} MB')

        total_elapsed = time.time() - overall_start
        m, s = divmod(int(total_elapsed), 60)
        print(f'\n{'='*55}')
        print(f'✅ All pairs merged! Total time: {m}m {s}s')
        print(f'   Output folder: {STATE["output_folder"]}')

    merge_btn.disabled = False

merge_btn.on_click(run_merge)
display(merge_btn, merge_out)

Button(button_style='primary', description='🎬 Merge All Pairs', layout=Layout(width='200px'), style=ButtonStyl…

Output()